# Week 10, Day 4 Lab — Multi-Agent Systems & Guardrails
### From one agent to a team — safely

This is the Week 10 capstone. Today you build a real **3-agent pipeline** (Research → Write →
Review) using plain Python functions and the free Gemini API — no new framework installs needed,
since handoffs and guardrails are framework-agnostic concepts (your own Day 4 slides show this
exact pattern in plain Python). Then you add the two guardrails from today's lecture: a
**permission boundary** and a **human-in-the-loop checkpoint**.

**How to use this notebook:**
- **WRITE THIS** cells have only a spec — you write the body.
- **TODO** cells ask you to modify or extend something already there.
- Plain cells are given — run them as-is.




---
## Section 0 — Setup

Just the free Gemini SDK today — no LangChain/LangGraph/CrewAI needed. Multi-agent handoffs
and guardrails are patterns you can build with plain functions, as you saw in today's slides.


In [ ]:
!pip install -q google-genai


In [ ]:
import os
import getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")


In [ ]:
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-2.5-flash"

def ask(prompt: str) -> str:
    """A tiny helper: send a prompt, get back plain text. This is the only building
    block every agent below is made of."""
    response = client.models.generate_content(model=MODEL, contents=prompt)
    return response.text

print(ask("Say 'Day 4 lab is ready!'"))


---
## Section 1 — Warm-Up: A Simple Handoff

A handoff can be as simple as "call function B with function
A's return value." Fully worked, using real Gemini calls instead of the placeholder `llm.chat`
from the slide.


In [ ]:
# GIVEN
def research_agent(topic: str) -> dict:
    findings = ask(f"Research the topic '{topic}'. List exactly 3 key facts, one per line, no extra commentary.")
    return {"topic": topic, "findings": findings}

def writer_agent(research_output: dict) -> str:
    prompt = (
        f"Write a short, friendly 3-sentence summary about {research_output['topic']} "
        f"using these findings:\n{research_output['findings']}"
    )
    return ask(prompt)


# The handoff: research_agent's output becomes writer_agent's input
findings = research_agent("Lahore Fort")
report = writer_agent(findings)
print(report)


What exactly did `writer_agent` need from `research_agent`'s output? What
would happen if you handed it the *entire* conversation history instead of just `findings`?


In [ ]:
# answer here

---
## Section 2 — Write Yourself: Add a Reviewer + Rejection Loop

A 2-agent pipeline has no guardrail at all — whatever the writer produces goes straight out.
Today's core pattern is a **reviewer** that can reject bad output and send it back. This
rejection loop *is* a guardrail: nothing reaches "final" without passing a check.

**Spec for `reviewer_agent`:**
- Signature: `reviewer_agent(draft: str) -> dict`
- Ask the model to review the draft for accuracy and completeness, with instructions like:
  *"If it's good, start your reply with the single word APPROVED. Otherwise start with REVISE
  and explain what to fix."*
- Return `{"approved": <bool>, "feedback": <the model's full reply>}` — `approved` is `True`
  only if the reply starts with "APPROVED" (case-insensitive)


In [ ]:
# WRITE THIS: reviewer_agent(draft) -> dict, per the spec above

def reviewer_agent(draft: str) -> dict:
    pass


# Quick manual test
print(reviewer_agent("Lahore Fort is a hotel located in Karachi built in 1990."))


**Spec for the pipeline function `run_pipeline`:**
- Signature: `run_pipeline(topic: str, max_revisions: int = 3) -> str`
- Call `research_agent`, then `writer_agent` to get an initial draft
- Call `reviewer_agent(draft)`. While it's not approved AND you haven't hit `max_revisions`:
  - print the feedback (so you can see *why* it was rejected)
  - ask `writer_agent` to revise — pass it a dict with the original findings **and** the
    feedback (you'll need to update `writer_agent`'s prompt to use feedback when present)
  - review the new draft again
- Return the final draft (approved, or the last attempt if `max_revisions` was reached)

**Why `max_revisions` matters:** without it, a reviewer that never approves creates an
infinite loop — the exact same failure mode as an agent with no `max_steps` cap from Day 1–2.


In [ ]:
# WRITE THIS: update writer_agent to accept optional feedback, and write run_pipeline

def writer_agent(research_output: dict) -> str:
    feedback = research_output.get("feedback")
    # if feedback is present, include it in the prompt so the writer can act on it
    pass


def run_pipeline(topic: str, max_revisions: int = 3) -> str:
    pass


final_report = run_pipeline("the history of Lahore Fort")
print()
print("=" * 50)
print("FINAL REPORT:", final_report)


 **Checkpoint :** temporarily set `max_revisions=0` and re-run on a fresh topic.
Confirm your pipeline still returns *something* (the un-reviewed first draft) instead of
crashing or hanging — a guardrail should fail safely, not fail loudly.


In [ ]:
#  test max_revisions=0 behaves safely
result = run_pipeline("a random topic of your choice", max_revisions=0)
print(result)


---
## Section 3 — Guardrail 1: Permission Boundaries

Right now every agent could call any function in your notebook — nothing stops the `writer`
from calling a "delete" tool if it wanted to. Let's build the least-privilege pattern from
your slides: each role gets its own scoped set of tools.

**GIVEN — two real tools, one safe and one dangerous:**


In [ ]:
# GIVEN
def save_report(text: str) -> dict:
    with open("report.txt", "w") as f:
        f.write(text)
    return {"status": "saved", "path": "report.txt"}

def delete_report() -> dict:
    import os
    if os.path.exists("report.txt"):
        os.remove("report.txt")
        return {"status": "deleted"}
    return {"status": "nothing to delete"}

tool_registry = {"save_report": save_report, "delete_report": delete_report}


**Spec for `make_scoped_registry` :**
- Signature: `make_scoped_registry(agent_role: str) -> dict`
- Define a `permissions` dict: `"writer"` can use `{"save_report"}`, `"reviewer"` gets `set()`
  (text-only role, no tool access), `"admin"` can use `{"save_report", "delete_report"}`
- Return only the entries of `tool_registry` whose name is in that role's allowed set


In [ ]:
# WRITE THIS: make_scoped_registry(agent_role) -> dict, per the spec above

def make_scoped_registry(agent_role: str) -> dict:
    pass


writer_tools = make_scoped_registry("writer")
reviewer_tools = make_scoped_registry("reviewer")

print("Writer can access:", list(writer_tools.keys()))
print("Reviewer can access:", list(reviewer_tools.keys()))


 **Checkpoint:** prove the boundary actually blocks something. Try to call
`delete_report` through `writer_tools` and confirm it raises a `KeyError` (the tool simply
isn't in that role's registry) rather than silently succeeding.


In [ ]:
#  confirm the writer role CANNOT reach delete_report
try:
    writer_tools["delete_report"]()
    print("PROBLEM: the writer was able to delete something it shouldn't have access to!")
except KeyError:
    print("Correctly blocked: 'delete_report' is not in the writer's scoped registry.")


---
## Section 4 — Guardrail 2: Human-in-the-Loop

Some actions are risky enough that even an allowed tool shouldn't run without a human okaying
it first. We'll treat "publishing" the final report as that kind of action.

**Spec for `execute_with_approval`:**
- Signature: `execute_with_approval(action_name: str, args: dict, high_risk_actions: set,
  tool_registry: dict) -> dict`
- If `action_name` is in `high_risk_actions`:
  - print a warning showing the action and its args
  - use `input("Approve this action? [y/N]: ")` to ask
  - if the answer isn't `"y"` (case-insensitive), return
    `{"status": "blocked", "reason": "Not approved by user"}` **without running the tool**
- Otherwise (or if approved), call `tool_registry[action_name](**args)` and return its result


In [ ]:
# WRITE THIS: execute_with_approval(...), per the spec above

def execute_with_approval(action_name: str, args: dict, high_risk_actions: set, tool_registry: dict) -> dict:
    pass


# Test it — this SHOULD pause and ask you to approve, since "save_report" is high-risk here
result = execute_with_approval(
    "save_report", {"text": final_report},
    high_risk_actions={"save_report"},
    tool_registry=tool_registry
)
print(result)


  **Checkpoint :** run the cell above twice — once typing `y` (approve) and once
typing anything else (reject). Confirm the file is only actually written when you approve.


---
## Section 5 — Independent Challenge: Put It All Together

Combine everything from Sections 2–4 into one run: the full research → write → review pipeline,
with the writer's tool access scoped via `make_scoped_registry`, and the final "publish" step
gated behind `execute_with_approval`.


In [ ]:
#  WRITE THIS: run the full pipeline end to end, then require approval to "publish"
# (save) the final report. Use a topic of your choice.

topic = "____________"
report = run_pipeline(topic)

writer_tools = make_scoped_registry("writer")
result = execute_with_approval(
    "save_report", {"text": report},
    high_risk_actions={"save_report", "delete_report"},
    tool_registry=writer_tools
)
print(result)


**Stretch goal (if time remains):** pick a deliberately vague or contradictory topic that's
likely to make the reviewer keep rejecting the draft, and confirm your `max_revisions` cap
still stops the loop cleanly instead of hanging.
